# Part 8: Save, load and share

A fine-tuned model is a folder: the weights, a configuration file and the tokenizer files. Anyone can load it from that folder, or from the Hub if you upload it. This part trains a very small model (32 examples, one epoch) because the point is the files, not the score.

In [ ]:
import warnings

from transformers.utils import logging

logging.set_verbosity_error()  # hide the long model-loading reports
warnings.filterwarnings("ignore")

from review_data import as_lists, load_reviews, small_sample

reviews = load_reviews()
val_texts, val_labels = as_lists(reviews["validation"])

texts, labels = small_sample(reviews["train"], 32, seed=42)

In [ ]:
from finetune import fine_tune
from share import save_model

model, tokenizer, _ = fine_tune(texts, labels, val_texts[:32], val_labels[:32], epochs=1)
folder = save_model(model, tokenizer, "saved_model")
print(sorted(p.name for p in folder.iterdir()))

In [ ]:
from share import label_scores, load_classifier

classifier = load_classifier("saved_model")
print(label_scores(classifier, "a warm, funny and beautifully acted film"))

## Run the Gradio app

From a terminal in this folder, run `python app.py saved_model`. Gradio prints a local address. Open it, type a sentence, and the app returns the two scores. The same `app.py` runs on a Space once the model is on the Hub.

## Share on the Hub

These two lines upload the model. They need a Hugging Face account and a token with write access, which you create in your account settings and store with `hf auth login`. Never paste a token into a notebook or commit it to Git.

```python
model.push_to_hub("your-username/review-sentiment")
tokenizer.push_to_hub("your-username/review-sentiment")
```

Then write a **model card**: what the model is for, the data it was trained on, how it was evaluated, and where it fails. The Hub shows the card on the model page.

## Try it

1. List the files in `saved_model` and look at `config.json`. Where are the label names?
2. Load the saved folder with `AutoModelForSequenceClassification` instead of a pipeline.
3. Write three sentences for a model card for this model, including one honest limitation.